# Phase S1 v3 — BN Only (Minimal)
Validate BatchNorm cooling: φ_BN ≈ 0.66

**Only BN D=48,96 × 2 seeds — ~2 hours**
Epochs: 300 (same as running version)

## ▶️ Step 1: Clone

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## ▶️ Step 2: Install Dependencies

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN')

!pip install scipy --quiet
import os, math, json, time, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## ▶️ Step 3: Training (BN D=48,96 × 2 seeds)

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
CONFIGS = [
    ('BN_NoSkip', 'batchnorm', [42, 123]),
]
D_VALUES = [48, 96]  # Only key points
EPOCHS = 300         # Same as running version
BATCH_SIZE = 128
LR = 0.01
WD = 5e-4
MOMENTUM = 0.9

OUT_DIR = Path('./phase_s1_bn_results')
OUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)
GAMMA_TRACK_EVERY = [50, 100, 150, 200, 250, 300]

# ── NETWORK ─────────────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, stride=1):
        super().__init__()
        self.norm_type = norm_type
        self.use_skip = use_skip
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        if norm_type == 'layernorm':
            self.norm = nn.LayerNorm([out_ch, 32, 32])
        elif norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        else:
            self.norm = nn.Identity()
        self.act = nn.GELU()
        if use_skip and in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        elif use_skip:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = None

    def forward(self, x):
        out = self.norm(self.conv(x))
        out = self.act(out)
        if self.use_skip and self.skip is not None:
            out = out + self.skip(x)
        return out

class ValidationNet(nn.Module):
    def __init__(self, base_ch=64, norm_type='none', use_skip=False, n_classes=10):
        super().__init__()
        channels = [3, base_ch, base_ch*2, base_ch*2]
        self.blocks = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.blocks.append(ConvBlock(
                channels[i], channels[i+1],
                norm_type=norm_type,
                use_skip=(i > 0 and use_skip),
                stride=1
            ))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels[-1], n_classes)

    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ── J_TOPO ──────────────────────────────────────────────────────────────────
from torch import linalg

def compute_D_eff(W):
    if W.dim() == 4:
        W = W.view(W.size(0), -1)
    fro_sq = (W ** 2).sum().item()
    S = linalg.svd(W.to('cpu')).S
    spec_sq = S[0].item() ** 2 + 1e-12
    return fro_sq / spec_sq

def compute_J_topo(weights, d_input=3.0):
    eta_vals = []
    d_prev = d_input
    for W in weights:
        if W.dim() == 4:
            W = W.view(W.size(0), -1)
        D_eff = compute_D_eff(W)
        eta = D_eff / max(d_prev, 1e-8)
        eta_vals.append(max(eta, 1e-8))
        d_prev = D_eff
    L = len(eta_vals)
    log_sum = sum(abs(math.log(e)) for e in eta_vals)
    return math.exp(-log_sum / L) if L > 0 else 0.0

# ── VARIANCE TRACKER ─────────────────────────────────────────────────────────
class VarianceTracker:
    def __init__(self, model):
        self.model = model
        self.activations = {}
        self.handles = []
        self._register_hooks()

    def _register_hooks(self):
        def get_activation(name):
            def hook(module, input, output):
                self.activations[name] = output.detach()
            return hook
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                self.handles.append(module.register_forward_hook(get_activation(name)))

    def get_variances(self):
        return {name: acts.var().item() for name, acts in self.activations.items()}

    def compute_gamma(self, init_variances):
        final_vars = self.get_variances()
        gamma_total = 0.0
        count = 0
        for name in init_variances:
            if name in final_vars:
                sigma_init = math.sqrt(init_variances[name])
                sigma_final = math.sqrt(final_vars[name])
                if sigma_init > 1e-8 and sigma_final > 1e-8:
                    gamma_total += abs(math.log(sigma_final / sigma_init))
                    count += 1
        return gamma_total / max(count, 1)

    def close(self):
        for h in self.handles:
            h.remove()

def measure_init_variance(model, batch_size=32):
    model = model.to(DEVICE)
    model.eval()
    tracker = VarianceTracker(model)
    dummy = torch.randn(batch_size, 3, 32, 32).to(DEVICE)
    with torch.no_grad():
        model(dummy)
    init_vars = tracker.get_variances()
    tracker.close()
    return init_vars

# ── DATALOADER ────────────────────────────────────────────────────────────────
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10(root='./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10(root='./data', train=False, transform=transform_val, download=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Data loaded: {len(train_ds)} train, {len(val_ds)} val")

In [ ]:
# ── TRAIN ────────────────────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs, lr, wd, momentum,
                init_variances=None, track_gamma=False, start_epoch=0):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_epoch = 0
    gamma_history = []

    tracker = None
    if track_gamma and init_variances is not None:
        tracker = VarianceTracker(model)

    t0 = time.time()
    pbar = tqdm(range(epochs), desc="Training", leave=False)

    for epoch in pbar:
        if epoch < start_epoch:
            scheduler.step()
            continue

        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)

        val_loss /= total
        val_acc = correct / total

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch + 1

        # Gamma tracking
        if tracker and ((epoch + 1) in GAMMA_TRACK_EVERY or epoch == epochs - 1):
            model.eval()
            dummy_x = torch.randn(64, 3, 32, 32).to(DEVICE)
            with torch.no_grad():
                model(dummy_x)
            gamma = tracker.compute_gamma(init_variances)
            gamma_history.append({'epoch': epoch + 1, 'gamma': gamma})

        elapsed = (time.time() - t0) / 60
        pbar.set_postfix({
            'loss': f'{val_loss:.4f}',
            'best': f'{best_val_loss:.4f}',
            'elapsed': f'{elapsed:.1f}m'
        })

        # Checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0 and epoch < epochs - 1:
            ckpt = {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }
            torch.save(ckpt, CKPT_DIR / 'current_ckpt.pt')

    if tracker:
        tracker.close()

    return {
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
        'final_val_loss': val_loss,
        'final_val_acc': val_acc,
        'gamma_history': gamma_history,
    }


def fit_scaling_law(losses_by_d, d_values):
    from scipy.optimize import curve_fit
    def power_law(D, alpha, beta, E):
        return alpha * np.array(D) ** (-beta) + E
    Ds = np.array(d_values)
    losses = np.array([losses_by_d[d] for d in d_values])
    try:
        popt, _ = curve_fit(power_law, Ds, losses, p0=[10.0, 0.5, 0.5],
                           bounds=([0, 0, 0], [1000, 5, 10]), maxfev=10000)
        alpha, beta, E = popt
        preds = power_law(Ds, *popt)
        ss_res = ((losses - preds) ** 2).sum()
        ss_tot = ((losses - losses.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return {'alpha': float(alpha), 'beta': float(beta), 'E': float(E), 'R2': float(r2)}
    except:
        return {'alpha': None, 'beta': None, 'E': None, 'R2': 0.0}


def estimate_params(base_ch):
    conv_params = 3*base_ch*9 + base_ch*base_ch*2*9 + base_ch*base_ch*2*9
    fc_params = 2*base_ch * 10
    return (conv_params + fc_params) / 1e6

In [ ]:
# ── MAIN LOOP ────────────────────────────────────────────────────────────────
from datetime import datetime

print("=" * 70)
print("Phase S1 v3 — BN Only (Minimal)")
print(f"Device: {DEVICE} | Epochs: {EPOCHS} | D: {D_VALUES}")
print("=" * 70)

# Load metadata
meta_file = OUT_DIR / 'metadata.json'
if meta_file.exists():
    with open(meta_file) as f:
        metadata = json.load(f)
else:
    metadata = {'completed_runs': []}

completed = set(metadata['completed_runs'])
total_runs = sum(len(cfg[2]) * len(D_VALUES) for cfg in CONFIGS)
print(f"Training {total_runs} runs | Epochs: {EPOCHS} | D: {D_VALUES}")

all_results = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'total_runs': total_runs,
    'configs': []
}

t_start = time.time()

for config_name, norm_type, seeds in CONFIGS:
    cfg_start = time.time()
    print(f"\n--- {config_name} ---")

    # J_topo
    model_init = ValidationNet(base_ch=64, norm_type=norm_type, use_skip=False).to(DEVICE)
    weights = model_init.get_conv_weights()
    J_topo = compute_J_topo(weights)
    del model_init
    torch.cuda.empty_cache()
    print(f"  J_topo_init = {J_topo:.4f}")

    cfg_result = {
        'name': config_name,
        'norm': norm_type,
        'J_topo_init': float(J_topo),
        'D_results': {}
    }

    for base_ch in D_VALUES:
        n_params = estimate_params(base_ch)
        d_result = {'base_ch': base_ch, 'n_params_M': float(n_params),
                    'seeds': {}, 'avg_val_loss': None, 'avg_gamma': None}
        losses = []
        gammas = []

        for seed in seeds:
            run_name = f"{config_name}_ch{base_ch}_s{seed}"
            ckpt_path = CKPT_DIR / f"{run_name}.pt"

            # Skip if completed
            if run_name in completed:
                if ckpt_path.exists():
                    ckpt = torch.load(ckpt_path, map_location=DEVICE)
                    result = ckpt['result']
                    losses.append(result['best_val_loss'])
                    if result.get('gamma_history'):
                        gammas.append(np.mean([g['gamma'] for g in result['gamma_history']]))
                    print(f"  D={base_ch} seed={seed} — loaded from checkpoint")
                continue

            # Train
            print(f"  D={base_ch} seed={seed} ({n_params:.1f}M) ... ", end='', flush=True)
            t0 = time.time()

            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            model = ValidationNet(base_ch=base_ch, norm_type=norm_type, use_skip=False).to(DEVICE)
            init_vars = measure_init_variance(model, batch_size=64)

            # Resume
            start_epoch = 0
            if ckpt_path.exists():
                ckpt = torch.load(ckpt_path, map_location=DEVICE)
                model.load_state_dict(ckpt['model_state'])
                start_epoch = ckpt.get('epoch', 0) + 1
                print(f"resuming from epoch {start_epoch} ... ", end='', flush=True)

            result = train_model(
                model, train_loader, val_loader,
                epochs=EPOCHS, lr=LR, wd=WD, momentum=MOMENTUM,
                init_variances=init_vars, track_gamma=True,
                start_epoch=start_epoch
            )

            elapsed = (time.time() - t0) / 60
            torch.save({'epoch': EPOCHS-1, 'model_state': model.state_dict(), 'result': result}, ckpt_path)

            avg_g = np.mean([g['gamma'] for g in result['gamma_history']]) if result.get('gamma_history') else 0
            print(f"loss={result['best_val_loss']:.4f} acc={result['best_val_acc']:.4f} γ={avg_g:.3f} [{elapsed:.0f}m]")

            gammas.append(avg_g)
            losses.append(result['best_val_loss'])
            d_result['seeds'][str(seed)] = result
            completed.add(run_name)
            metadata['completed_runs'] = list(completed)
            with open(meta_file, 'w') as f:
                json.dump(metadata, f)
            del model; torch.cuda.empty_cache()

        if losses:
            d_result['avg_val_loss'] = float(np.mean(losses))
            d_result['avg_gamma'] = float(np.mean(gammas)) if gammas else None
        cfg_result['D_results'][str(base_ch)] = d_result

    # Fit scaling law
    losses_by_d = {ch: cfg_result['D_results'][str(ch)]['avg_val_loss']
                   for ch in D_VALUES if cfg_result['D_results'][str(ch)]['avg_val_loss'] is not None}
    if losses_by_d:
        fit = fit_scaling_law(losses_by_d, list(losses_by_d.keys()))
        cfg_result['scaling_fit'] = fit
        print(f"  [{config_name}] β={fit['beta']:.4f} R²={fit['R2']:.4f} (wall: {(time.time()-cfg_start)/60:.0f}min)")

    all_results['configs'].append(cfg_result)

# Save
out_file = OUT_DIR / 'phase_s1_bn_results.json'
with open(out_file, 'w') as f:
    json.dump(all_results, f, indent=2)

# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"\n{'Config':<15} {'β':<10} {'γ':<10}")
print("-" * 40)
for cfg in all_results['configs']:
    fit = cfg.get('scaling_fit', {})
    beta = fit.get('beta') or 0
    gammas = [cfg['D_results'][str(ch)]['avg_gamma']
              for ch in D_VALUES if cfg['D_results'][str(ch)]['avg_gamma'] is not None]
    avg_gamma = sum(gammas)/len(gammas) if gammas else 0
    print(f"{cfg['name']:<15} {beta:<10.4f} {avg_gamma:<10.4f}")

print(f"\nTotal runtime: {(time.time()-t_start)/60:.1f} min")
print(f"Results: {out_file}")